# Lesson 08: Building Generative AI Chat Applications

In this notebook, you will:
1. Build a stateful multi-turn chat using `client.chats.create()` and the 4-part System Message Framework.
2. Run `tl;dr` summarization, zero-shot ticket classification, and few-shot product naming.
3. Compute `gemini-embedding-001` cosine similarity across words (`automobile`, `vehicle`, `dinosaur`, `stick`) and 3 news articles.

In [ ]:
!pip install -q -U google-genai numpy

In [ ]:
import os, numpy as np
from google import genai
from google.genai import types

def get_client():
    key = None
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass
    key = key or os.environ.get("GEMINI_API_KEY")
    return genai.Client(api_key=key) if key else None

client = get_client()

In [ ]:
if client:
    chat = client.chats.create(
        model="gemini-2.5-flash",
        config=types.GenerateContentConfig(
            system_instruction="You are a concise writing coach. Answer in 2 sentences max with clear examples."
        ),
    )
    print("Chat Turn 1:", chat.send_message("Should oxford commas always be used?").text)

    # Zero-shot classification
    clf = client.models.generate_content(
        model="gemini-2.5-flash",
        contents="Classify into [Pricing, Hardware Support, Software Support]: 'one of the keys on my laptop keyboard broke'",
    )
    print("Ticket Category:", clf.text.strip())

    # Embeddings cosine similarity
    words = ["automobile", "vehicle", "dinosaur", "stick"]
    embs = client.models.embed_content(model="gemini-embedding-001", contents=words)
    vecs = [np.array(e.values) for e in embs.embeddings]
    cos = lambda a, b: float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
    print(f"Sim(automobile, vehicle)={cos(vecs[0], vecs[1]):.3f} | Sim(automobile, dinosaur)={cos(vecs[0], vecs[2]):.3f}")
else:
    print("[Offline Demo Mode] Ticket Category: Hardware Support | Sim(automobile, vehicle) > Sim(automobile, dinosaur)")